# AIDEN Agent Training — Google Colab

Train all 5 AIDEN agents using LoRA + 4-bit QLoRA on a free T4 GPU.

**⏱ Total time:** ~30-60 min (depends on dataset size and epochs)
**💾 Disk:** ~5 GB for model downloads
**🧠 GPU:** T4 (free tier) sufficient for batch=1-2

**Instructions:** Run each cell in order. Each section is clearly labeled.

## Step 1 — Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Setup project directory
PROJECT_DIR = "/content/drive/MyDrive/aiden_project"
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}

# Clone or pull
if not os.path.exists("aiden"):
    !git clone https://github.com/Bharath8220818/aiden.git
else:
    %cd aiden
    !git pull
    %cd ..

%cd aiden/backend
print(f"\nWorking directory: {os.getcwd()}")

## Step 2 — Install Dependencies

In [ ]:
# Install core training dependencies
!pip install -q transformers>=4.56.2 torch accelerate peft bitsandbytes trl datasets sentencepiece huggingface_hub

# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime > Change runtime type > GPU")

# IMPORTANT: Accept the Llama-3.2 license before proceeding!
# Visit: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
# Click 'Agree and access repository' — without this, model download will fail with 403

# HuggingFace login (required for Llama-3.2-3B-Instruct)
import getpass
from huggingface_hub import login
HF_TOKEN = getpass.getpass("Enter your HuggingFace token (get from huggingface.co/settings/tokens): ")
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN
print("HuggingFace authenticated!")

## Step 3 — Verify Datasets

In [ ]:
!echo "=== Available Datasets ===" && ls -la data/*.jsonl
!echo "\n=== Dataset Line Counts ===" && wc -l data/*.jsonl

## Step 4 — Train Intent Agent

Natural language → structured pipeline configuration (JSON)

In [ ]:
!python scripts/train_agent.py \
    --agent intent \
    --data data/intent_dataset.jsonl \
    --epochs 3 \
    --batch 2 \
    --output models/adapters

## Step 5 — Train Extraction Agent

Schema discovery and data extraction from source systems

In [ ]:
!python scripts/train_agent.py \
    --agent extraction \
    --data data/extraction_dataset.jsonl \
    --epochs 3 \
    --batch 1 \
    --output models/adapters

## Step 6 — Train Monitoring Agent

Pipeline health monitoring and alert generation

In [ ]:
!python scripts/train_agent.py \
    --agent monitoring \
    --data data/monitoring_dataset.jsonl \
    --epochs 3 \
    --batch 2 \
    --output models/adapters

## Step 7 — Train Self-Healing Agent

Failure detection and automatic pipeline repair

In [ ]:
!python scripts/train_agent.py \
    --agent self_healing \
    --data data/self_healing_dataset.jsonl \
    --epochs 3 \
    --batch 1 \
    --output models/adapters

## Step 8 — Train Pipeline Builder Agent

Generate executable pipeline code (Airflow DAGs, dbt models)

In [ ]:
!python scripts/train_agent.py \
    --agent pipeline_builder \
    --data data/pipeline_builder_dataset.jsonl \
    --epochs 3 \
    --batch 1 \
    --output models/adapters

## Step 9 — Verify All Trained Adapters

In [ ]:
import os

adapter_dirs = [
    "models/adapters/intent-parser",
    "models/adapters/extraction",
    "models/adapters/monitoring",
    "models/adapters/self-healing",
    "models/adapters/pipeline-builder",
]

print("=== Trained Adapters ===")
for d in adapter_dirs:
    if os.path.exists(d):
        size = sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)) / 1024 / 1024
        files = os.listdir(d)
        print(f"  ✅ {d} ({size:.1f} MB, {len(files)} files)")
    else:
        print(f"  ❌ {d} — NOT FOUND")

## Step 10 — Backup Adapters to Google Drive

In [ ]:
import shutil

BACKUP_DIR = "/content/drive/MyDrive/aiden_models"
os.makedirs(BACKUP_DIR, exist_ok=True)

for adapter in adapter_dirs:
    if os.path.exists(adapter):
        name = os.path.basename(adapter)
        dest = os.path.join(BACKUP_DIR, name)
        shutil.copytree(adapter, dest, dirs_exist_ok=True)
        print(f"  ✅ Backed up: {name}")

print(f"\nAll adapters backed up to: {BACKUP_DIR}")

## Step 11 — Download to Local Machine

In [ ]:
from google.colab import files

# Zip all adapters (use full path to avoid shell cd issues)
!zip -r /content/all_adapters.zip models/adapters/

# Download
files.download('/content/all_adapters.zip')
print("\nDownload started! Extract to backend/models/adapters/ on your local machine.")

## Step 12 — Test Intent Parser (Optional)

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/aiden_project/aiden/backend')

from app.core.intent_parser import IntentParser
import asyncio

async def test():
    parser = IntentParser()
    result = await parser.parse("Build a daily sales ETL from PostgreSQL to Snowflake")
    print("Parsed result:")
    print(result)

asyncio.run(test())